In [2]:
import torch 
from transformers import pipeline
import numpy as np

model = f"cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_task = pipeline("sentiment-analysis", model=model, truncation=True, max_length=512)

result = sentiment_task("We played really well today and the team showed great character")                                                                                  
print(result)   

/Users/shakurahmad/PythonProjects/pl-press-conference-analyser/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassi

[{'label': 'positive', 'score': 0.9857186675071716}]


In [3]:
result = sentiment_task("We need to be more aggressive and press them higher up the pitch")                                                                                 
print(result)

[{'label': 'neutral', 'score': 0.7723646759986877}]


In [8]:
result = sentiment_task("We were poor today, we didn't compete and the performance was not acceptable")                                                                     
print(result)   

0.9414147734642029


In [10]:
import sqlite3

conn = sqlite3.connect('/Users/shakurahmad/PythonProjects/pl-press-conference-analyser/data/database.db')

cur = conn.cursor()

cur.execute("SELECT * FROM arsenal LIMIT 1")

row = cur.fetchone()    # get the result
body_text = row[4]      # body is the 5th column (index 4)
result = sentiment_task(body_text) 
print(result)
print(body_text)

[{'label': 'neutral', 'score': 0.5386993885040283}]
When Alexis’ heavily-deflected free kick flew in with seven minutes remaining at West Brom, we looked to be on our way to a hard-fought win. But a controversial handball decision against Calum Chambers in the closing moments allowed the hosts to equalise from the spot and deny us the three points. Arsene Wenger was unhappy at the incident, and this is what he said in his post-match press conference: on the penalty decision… I think the referee has not even seen it. He has not seen it. on why he was so angry… I’m angry because we have seen the same things again and again and I said down there that we have fought very hard for the referees to become professional many years ago with David Dein and we did a good job to allow them to become professional and I see no improvement. At the end of the day there are two countries in Europe where there are professional referees, in Italy and in England, but not one English referee will go to the 

This is very revealing. If i take a snippet from the text body: 

Arsene Wenger was unhappy at the incident, and this is what he said in his post-match press conference: on the penalty decision… I think the referee has not even seen it. He has not seen it. on why he was so angry…   

I’m angry because we have seen the same things again and again and I said down there that we have fought very hard for the referees to become professional many years ago     
with David Dein and we did a good job to allow them to become professional and I see no improvement.


It is very clearly negative. Neutral does not make sense here and a confidence level of 0.54 shows that it is not reliable. The transcript includes a lot of natural-sounding factual language mixed in with the anger - "we played", "the referee", "west Brom with five days to prepare". The model is proabbly averaging across all of that.

This shows the model is struggling with football-domain language. 

Let's check the sentiment prediction after a big loss:

In [5]:
cur.execute("SELECT body FROM arsenal ORDER BY created DESC LIMIT 1")

lost_match_body = cur.fetchone()

pred = sentiment_task(lost_match_body[0], truncation=True)
print(pred)


[{'label': 'negative', 'score': 0.6359798312187195}]


It gave the correct direction but still not very confident. And this this after a defeat in a cup final so we would expect high confidence. Instead on tuncation we will look at chunking for a long term solution. 

Chunking has now been implemented, splitting files into question and answer format. It will be investigated to see if this improves accuracy and confidence.

In [10]:
import sys
sys.path.append('/Users/shakurahmad/PythonProjects/pl-press-conference-analyser')
from src.preprocessing.chunking import chunk_by_qa

body = "<p>Unai Emery was a bundle of energy on touchline throughout his first Premier League game as our head coach. But he couldn\u2019t get the result he wanted against Manchester City.</p>\n<p>It was the toughest possible start to Unai\u2019s tenure and in his post-match press conference he picked out the positives and identified where we need to improve.</p>\n<p>Read on for a full transcript:</p>\n<p><strong>on his verdict after his first game...</strong></p>\n<p>First, it's a good ambience here with our supporters and also with a big motivation for us to start watching the squad and also we wanted to start today, here in the first match, with our supporters. Today, Manchester City's performance showed us that we need to continue in our process to improve. I think they deserved this result, but we were improving in the 90 minutes, like I think we need to do for the next week and the next match on the pitch. In the second half, it's the moment that maybe we had chances to get a better result.</p>\n<p><strong>on the players' application of his ideas...</strong></p>\n<p>Today, Manchester City demanded our best performance. Today we watched and we need to continue working. I am happy with the players because they ran and they fought. We need to continue working tactically and defensively, and doing more to shorten the differences today between Manchester City and us.</p>\n<p><strong>on this game being a barometer of what progress is needed...</strong></p>\n<p>Yes. But we know this. We're starting and we need also to do one process one way. Today is the first step. Manchester City are working in their third year with Guardiola and they have built a team with security, good players and a great stability of playing like they want. We are only starting out now.</p>\n<p><strong>on not picking Bernd Leno...</strong></p>\n<p>We want the competition between the goalkeepers and also every player in the squad. Despite the position, it is the same opportunity and the same chances for all the goalkeepers. We spoke also and Petr Cech, he is doing very well. Today as well, I think he played well also. He has this experience for continuing to defend our goal. Also, Leno is starting with us. He is working very well, he played well in pre-season too but he has to wait for his chance to arrive.</p>\n<p><strong>on the first goal being reminiscent of the old Arsenal...</strong></p>\n<p>It's two different halves, the first and the second half, for me. The first half, we conceded more space on the pitch for Manchester City to progress. They deal very well with these situations and because they know that, they have confidence in their performance. In the second half, we took more risks with our pressing, took more risks with the ball to break their lines and go forward quickly. We wanted to create a goal so that we could get back into the game. I want to continue the process to build our team. I think we finished with the spirit I want and the team, for 90 minutes, they ran, they tried and they pushed. But I think we need to improve collectively and also individually. But I think this process is normal today against a great team like Manchester City.</p>\n<p><strong>on facing Chelsea next week and how we avoid having our press broken...</strong></p>\n<p>Each match is different. We want to prepare for each match with the difficulty and demands of the opposition. But today, I think the team in more time, we are playing with this personality and we need to improve this personality with better performances on the pitch. In the next week we are continuing this work and against Chelsea we are going to analyse the opposition. We are going there to win, with this intention against Chelsea.</p>\n<p><strong>on Ainsley Maitland-Niles\u2026</strong></p>\n<p>Today we are going to do the medical analysis with the doctor. Nacho is starting with the group, working yesterday and maybe he can come in, maybe, for the next match. I think every day is important to know how they are.</p>\n"



test = sentiment_task(chunk_by_qa(body)[0]['answer'])
print(test)

7
{'question': 'on his verdict after his first game...', 'answer': "First, it's a good ambience here with our supporters and also with a big motivation for us to start watching the squad and also we wanted to start today, here in the first match, with our supporters. Today, Manchester City's performance showed us that we need to continue in our process to improve. I think they deserved this result, but we were improving in the 90 minutes, like I think we need to do for the next week and the next match on the pitch. In the second half, it's the moment that maybe we had chances to get a better result."}
[{'label': 'positive', 'score': 0.9355828762054443}]


"Positive" at 94% for Emery's first game after losing to Man City is clearly wrong. The answer is quite neutral/measure, not positvie. 

This is the domain problem again. The model is picking up on words like "good ambience", "motivation", "improving" and classifying as positive, missing the context that tis is a manager making the best of a defeat. 

In [14]:
quote = "<p>When Alexis\u2019 heavily-deflected free kick flew in with seven minutes remaining at West Brom, we looked to be on our way to a hard-fought win.</p>\n<p>But a controversial handball decision against Calum Chambers in the closing moments allowed the hosts to equalise from the spot and deny us the three points.</p>\n<p>Arsene Wenger was unhappy at the incident, and this is what he said in his post-match press conference:</p>\n<p><strong>on the penalty decision\u2026</strong><br><br>\nI think the referee has not even seen it. He has not seen it.</p>\n<p><strong>on why he was so angry\u2026</strong><br><br>\nI\u2019m angry because we have seen the same things again and again and I said down there that we have fought very hard for the referees to become professional many years ago with David Dein and we did a good job to allow them to become professional and I see no improvement. At the end of the day there are two countries in Europe where there are professional referees, in Italy and in England, but not one English referee will go to the World Cup, but everything is alright. We cannot say a word against because they are untouchable, it is the case and the truth as well, it is a reason for it. It\u2019s not only me who judges them.</p>\n<p><strong>on the referee not seeing the handball\u2026</strong><br><br>\nThat is my opinion, he will tell you he has seen it.</p>\n<p><strong>on whether his assistant told him or he guessed\u2026</strong><br><br>\nI don\u2019t know, ask him. I don\u2019t want to waste my time and we have to live with the decision. You will go home and have your normal thing and we have to live with it and swallow it for the next game. There are two things that are not normal in the Premier League, the schedule and the referees. Maybe we haven\u2019t played well enough, that is down for you to judge, but with those two things you cannot have West Brom with five days to prepare while we have three. I am ready to play every day as long as our opponent has had the same recovery time over Christmas, it\u2019s not normal. We have the same problem against Chelsea, they played yesterday, we played today. They have one day more.</p>\n<p>&nbsp;</p>\n"

test_qa = sentiment_task(chunk_by_qa(quote)[0]['answer'])
print(test_qa)

4
{'question': 'on the penalty decision…', 'answer': 'on why he was so angry…I’m angry because we have seen the same things again and again and I said down there that we have fought very hard for the referees to become professional many years ago with David Dein and we did a good job to allow them to become professional and I see no improvement. At the end of the day there are two countries in Europe where there are professional referees, in Italy and in England, but not one English referee will go to the World Cup, but everything is alright. We cannot say a word against because they are untouchable, it is the case and the truth as well, it is a reason for it. It’s not only me who judges them.'}
[{'label': 'negative', 'score': 0.6324666142463684}]


The answer is clearly angry as the model has identified but the confidence level is still not the best. 

Search HuggingFace for sentiment models trained on financial news. The langauge is measured, formal, media trained, where emotions are expressed subtly. This will be investigated with "distilroberta-finetuned-financial-news-sentiment-analysis":

In [15]:
fin_model = pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

test_fin_model_qa = fin_model(chunk_by_qa(quote)[0]['answer'])
print(test_fin_model_qa)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/shakurahmad/PythonProjects/pl-press-conference-analyser/.venv/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


4
{'question': 'on the penalty decision…', 'answer': 'on why he was so angry…I’m angry because we have seen the same things again and again and I said down there that we have fought very hard for the referees to become professional many years ago with David Dein and we did a good job to allow them to become professional and I see no improvement. At the end of the day there are two countries in Europe where there are professional referees, in Italy and in England, but not one English referee will go to the World Cup, but everything is alright. We cannot say a word against because they are untouchable, it is the case and the truth as well, it is a reason for it. It’s not only me who judges them.'}
[{'label': 'negative', 'score': 0.8348150253295898}]


The model has improved here, it is now more confident with its negative rating.

In [17]:
test_fin_model_emry = fin_model(chunk_by_qa(body)[0]['answer'])
print(test_fin_model_emry)

7
{'question': 'on his verdict after his first game...', 'answer': "First, it's a good ambience here with our supporters and also with a big motivation for us to start watching the squad and also we wanted to start today, here in the first match, with our supporters. Today, Manchester City's performance showed us that we need to continue in our process to improve. I think they deserved this result, but we were improving in the 90 minutes, like I think we need to do for the next week and the next match on the pitch. In the second half, it's the moment that maybe we had chances to get a better result."}
[{'label': 'positive', 'score': 0.9992766976356506}]


The model has performed dissapointingly here as it gives an even more confident score than before, and still wrong. This tells us the both models struglle with the same type of text - measure, displomatic langauge where a manager is pussting a positive spin on a negative result. 

In [11]:
cur.execute("SELECT id, label, score FROM chunks LIMIT 5")
print(cur.fetchall())

[(1, 'negative', 0.6324666142463684), (2, 'neutral', 0.7431052327156067), (3, 'negative', 0.5589945912361145), (4, 'positive', 0.4012986123561859), (5, 'neutral', 0.6535377502441406)]


In [12]:
cur.execute("SELECT label, COUNT(*) FROM chunks GROUP BY label")
print(cur.fetchall())

[('negative', 539), ('neutral', 2807), ('positive', 2813)]


The negative count seems very low for press conferences that cover losses, injuries, and refree controversies. This again confirms the domain problem - the model is systemeatically classifying measure diplomatic language as neutral or positive rather than negative. 